# Clean Raw Data

Connects to Databricks via `databricks-connect` and gives you live access to the 4 raw Delta tables in `mlops_dev.chenheju` for cleaning and feature engineering.

In [1]:
import os

from databricks.connect import DatabricksSession

os.environ.setdefault("DATABRICKS_CONFIG_PROFILE", "llmops-course")

spark = DatabricksSession.builder.profile("llmops-course").getOrCreate()

print(f"Spark version : {spark.version}")
print("Connection OK")

Spark version : 4.1.0
Connection OK


In [2]:
CATALOG = "mlops_dev"
SCHEMA = "chenheju"

members = spark.table(f"{CATALOG}.{SCHEMA}.members")
train = spark.table(f"{CATALOG}.{SCHEMA}.train")
transactions = spark.table(f"{CATALOG}.{SCHEMA}.transactions")
user_logs = spark.table(f"{CATALOG}.{SCHEMA}.user_logs")

for name, df in [
    ("members", members),
    ("train", train),
    ("transactions", transactions),
    ("user_logs", user_logs),
]:
    print(f"{name}: {df.count():,} rows  |  columns: {df.columns}")

members: 6,769,473 rows  |  columns: ['msno', 'city', 'bd', 'gender', 'registered_via', 'registration_init_time']
train: 970,960 rows  |  columns: ['msno', 'is_churn']
transactions: 1,431,009 rows  |  columns: ['msno', 'payment_method_id', 'payment_plan_days', 'plan_list_price', 'actual_amount_paid', 'is_auto_renew', 'transaction_date', 'membership_expire_date', 'is_cancel']
user_logs: 18,396,362 rows  |  columns: ['msno', 'date', 'num_25', 'num_50', 'num_75', 'num_985', 'num_100', 'num_unq', 'total_secs']


## Your cleaning / feature engineering goes here

Work on each DataFrame and write back when ready:

```python
members_clean = members  # ... your transformations ...

(
    members_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.members")
)
```

## Processing Members Data

In [3]:
# Basic info
print("Row count:", members.count())
print("Columns:", members.columns)
members.printSchema()

Row count: 6769473
Columns: ['msno', 'city', 'bd', 'gender', 'registered_via', 'registration_init_time']
root
 |-- msno: string (nullable = true)
 |-- city: integer (nullable = true)
 |-- bd: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- registered_via: integer (nullable = true)
 |-- registration_init_time: integer (nullable = true)



In [4]:
members.show(5, truncate=False)

+--------------------------------------------+----+---+------+--------------+----------------------+
|msno                                        |city|bd |gender|registered_via|registration_init_time|
+--------------------------------------------+----+---+------+--------------+----------------------+
|GpWLe822TqXbhfUbifCnZoh3O8ckd7bCzaI1/iVk2y0=|1   |0  |NULL  |4             |20160111              |
|6AttH1N3YiLd2Lxg7f9MQ9OMWaD039shkjFrw93iS/I=|1   |0  |NULL  |4             |20160111              |
|Plc7zMIJlWh4v2mWz3DXCIEO9U4mSclREB8AOyIHLb8=|1   |0  |NULL  |4             |20160111              |
|wjSqudM35HWZoXE+0K1OAtOh/6JMUEOIL4pMft5GsVE=|1   |0  |NULL  |4             |20160111              |
|lLyu2yHb60gB0R+NGpVwPf+XI23kn28tVNvVnavmt60=|1   |23 |female|3             |20160111              |
+--------------------------------------------+----+---+------+--------------+----------------------+
only showing top 5 rows


In [5]:
from pyspark.sql.functions import col, when
from pyspark.sql.functions import sum as _sum

members.select(
    [_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in members.columns]
).show()

+----+----+---+-------+--------------+----------------------+
|msno|city| bd| gender|registered_via|registration_init_time|
+----+----+---+-------+--------------+----------------------+
|   0|   0|  0|4429505|             0|                     0|
+----+----+---+-------+--------------+----------------------+



In [6]:
4429505 / 6769473

0.6543352783887313

In [7]:
from pyspark.sql.functions import col, to_date

members = members.withColumn(
    "registration_init_time",
    to_date(col("registration_init_time").cast("string"), "yyyyMMdd"),
)

In [8]:
members.select("registration_init_time").show(5)
members.printSchema()

+----------------------+
|registration_init_time|
+----------------------+
|            2016-01-11|
|            2016-01-11|
|            2016-01-11|
|            2016-01-11|
|            2016-01-11|
+----------------------+
only showing top 5 rows
root
 |-- msno: string (nullable = true)
 |-- city: integer (nullable = true)
 |-- bd: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- registered_via: integer (nullable = true)
 |-- registration_init_time: date (nullable = true)



## Processing Train Data

In [9]:
print("Row count:", train.count())
print("Columns:", train.columns)
train.printSchema()

Row count: 970960
Columns: ['msno', 'is_churn']
root
 |-- msno: string (nullable = true)
 |-- is_churn: integer (nullable = true)



In [10]:
train.select(
    [_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in train.columns]
).show()

+----+--------+
|msno|is_churn|
+----+--------+
|   0|       0|
+----+--------+



In [11]:
train.show(5, truncate=False)

+--------------------------------------------+--------+
|msno                                        |is_churn|
+--------------------------------------------+--------+
|ugx0CjOMzazClkFzU2xasmDZaoIqOUAZPsH1q0teWCg=|1       |
|f/NmvEzHfhINFEYZTR05prUdr+E+3+oewvweYz9cCQE=|1       |
|zLo9f73nGGT1p21ltZC3ChiRnAVvgibMyazbCxvWPcg=|1       |
|8iF/+8HY8lJKFrTc7iR9ZYGCG2Ecrogbc2Vy5YhsfhQ=|1       |
|K6fja4+jmoZ5xG6BypqX80Uw/XKpMgrEMdG2edFOxnA=|1       |
+--------------------------------------------+--------+
only showing top 5 rows


## Processing Transactions Data

In [12]:
print("Row count:", transactions.count())
print("Columns:", transactions.columns)
transactions.printSchema()

Row count: 1431009
Columns: ['msno', 'payment_method_id', 'payment_plan_days', 'plan_list_price', 'actual_amount_paid', 'is_auto_renew', 'transaction_date', 'membership_expire_date', 'is_cancel']
root
 |-- msno: string (nullable = true)
 |-- payment_method_id: integer (nullable = true)
 |-- payment_plan_days: integer (nullable = true)
 |-- plan_list_price: integer (nullable = true)
 |-- actual_amount_paid: integer (nullable = true)
 |-- is_auto_renew: integer (nullable = true)
 |-- transaction_date: integer (nullable = true)
 |-- membership_expire_date: integer (nullable = true)
 |-- is_cancel: integer (nullable = true)



In [13]:
transactions.show(5, truncate=False)

+--------------------------------------------+-----------------+-----------------+---------------+------------------+-------------+----------------+----------------------+---------+
|msno                                        |payment_method_id|payment_plan_days|plan_list_price|actual_amount_paid|is_auto_renew|transaction_date|membership_expire_date|is_cancel|
+--------------------------------------------+-----------------+-----------------+---------------+------------------+-------------+----------------+----------------------+---------+
|D+i+7Z0gdSGmJE7lOsxuNU8XMKmxWUpx4NI2plf8naQ=|30               |30               |149            |149               |1            |20170301        |20170331              |0        |
|D/155dM1GgmoslzJFqebg2z/zTzzakbVE8hYXSvk3Jo=|32               |410              |1788           |1788              |0            |20161214        |20180202              |0        |
|D/bG4gqPUDNboEk9r+cPGDzI1ASbrSRclZT+SZq2VyE=|41               |30               |99      

In [14]:
transactions.select(
    [_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in transactions.columns]
).show()

+----+-----------------+-----------------+---------------+------------------+-------------+----------------+----------------------+---------+
|msno|payment_method_id|payment_plan_days|plan_list_price|actual_amount_paid|is_auto_renew|transaction_date|membership_expire_date|is_cancel|
+----+-----------------+-----------------+---------------+------------------+-------------+----------------+----------------------+---------+
|   0|                0|                0|              0|                 0|            0|               0|                     0|        0|
+----+-----------------+-----------------+---------------+------------------+-------------+----------------+----------------------+---------+



In [15]:
from pyspark.sql.functions import col, to_date

transactions = transactions.withColumn(
    "transaction_date", to_date(col("transaction_date").cast("string"), "yyyyMMdd")
).withColumn(
    "membership_expire_date",
    to_date(col("membership_expire_date").cast("string"), "yyyyMMdd"),
)

In [16]:
transactions.select("transaction_date", "membership_expire_date").show(5)
transactions.printSchema()

+----------------+----------------------+
|transaction_date|membership_expire_date|
+----------------+----------------------+
|      2017-03-01|            2017-03-31|
|      2016-12-14|            2018-02-02|
|      2017-03-31|            2017-04-30|
|      2017-03-31|            2017-05-06|
|      2017-03-17|            2017-04-16|
+----------------+----------------------+
only showing top 5 rows
root
 |-- msno: string (nullable = true)
 |-- payment_method_id: integer (nullable = true)
 |-- payment_plan_days: integer (nullable = true)
 |-- plan_list_price: integer (nullable = true)
 |-- actual_amount_paid: integer (nullable = true)
 |-- is_auto_renew: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- membership_expire_date: date (nullable = true)
 |-- is_cancel: integer (nullable = true)



## Process User_logs Dataset

In [17]:
print("Row count:", user_logs.count())
print("Columns:", user_logs.columns)
user_logs.printSchema()

Row count: 18396362
Columns: ['msno', 'date', 'num_25', 'num_50', 'num_75', 'num_985', 'num_100', 'num_unq', 'total_secs']
root
 |-- msno: string (nullable = true)
 |-- date: integer (nullable = true)
 |-- num_25: integer (nullable = true)
 |-- num_50: integer (nullable = true)
 |-- num_75: integer (nullable = true)
 |-- num_985: integer (nullable = true)
 |-- num_100: integer (nullable = true)
 |-- num_unq: integer (nullable = true)
 |-- total_secs: double (nullable = true)



In [18]:
user_logs.show(5, truncate=False)

+--------------------------------------------+--------+------+------+------+-------+-------+-------+----------+
|msno                                        |date    |num_25|num_50|num_75|num_985|num_100|num_unq|total_secs|
+--------------------------------------------+--------+------+------+------+-------+-------+-------+----------+
|31faHqRKF/Ty+vueH4TFkauvpAyyE3u5Dz71saGhA90=|20170309|0     |0     |0     |0      |134    |95     |32212.074 |
|wiZ8j97jpHUvf/XG6mfi8BEcTnsqfDkMej/7LijfIY0=|20170302|4     |0     |0     |1      |31     |26     |7909.499  |
|QAQqFd6SPPsStXsqqtVLdjpsmf04UoE6FBehQt7i3B0=|20170302|0     |1     |0     |0      |16     |16     |3959.94   |
|1buwtr3vVbdQB9ZuTWaMgtUT/XQRzOGw8yLjagMUITI=|20170302|0     |0     |1     |0      |86     |83     |21994.702 |
|DRDhL/F/fZKpvHKRwqrCXrt7py43aC5/SOJPdYRv0Y4=|20170302|111   |156   |103   |33     |90     |414    |58092.354 |
+--------------------------------------------+--------+------+------+------+-------+-------+-------+----

In [19]:
user_logs = user_logs.withColumn("date", to_date(col("date").cast("string"), "yyyyMMdd"))

In [20]:
user_logs.select("date").show(5)
user_logs.printSchema()

+----------+
|      date|
+----------+
|2017-03-09|
|2017-03-02|
|2017-03-02|
|2017-03-02|
|2017-03-02|
+----------+
only showing top 5 rows
root
 |-- msno: string (nullable = true)
 |-- date: date (nullable = true)
 |-- num_25: integer (nullable = true)
 |-- num_50: integer (nullable = true)
 |-- num_75: integer (nullable = true)
 |-- num_985: integer (nullable = true)
 |-- num_100: integer (nullable = true)
 |-- num_unq: integer (nullable = true)
 |-- total_secs: double (nullable = true)



In [21]:
user_logs.select(
    [_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in user_logs.columns]
).show()

+----+----+------+------+------+-------+-------+-------+----------+
|msno|date|num_25|num_50|num_75|num_985|num_100|num_unq|total_secs|
+----+----+------+------+------+-------+-------+-------+----------+
|   0|   0|     0|     0|     0|      0|      0|      0|         0|
+----+----+------+------+------+-------+-------+-------+----------+



## Update the 4 delta tables

In [22]:
tables = [
    ("members", members),
    ("train", train),
    ("transactions", transactions),
    ("user_logs", user_logs),
]

for table_name, df in tables:
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_name)
    )
    print(f"Done: {full_name}")

print("\nAll 4 tables updated successfully.")

Done: mlops_dev.chenheju.members
Done: mlops_dev.chenheju.train
Done: mlops_dev.chenheju.transactions
Done: mlops_dev.chenheju.user_logs

All 4 tables updated successfully.
